In [45]:
# this for CNN
# will use torch.nn.Conv2d and torchvision
import torch
from torch import nn

import torchvision
from torchvision import transforms
from torchvision import datasets
from torchvision.transforms import ToTensor

import matplotlib.pyplot as plt
print(torch.__version__)
print(torchvision.__version__)

2.6.0+cu124
0.21.0+cu124


In [46]:
# getting a dataset(fashion mnist)

train_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
    target_transform=None
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)


In [47]:
len(train_data)


60000

In [48]:
image, label = train_data[0]
label

9

In [49]:
len(image)

1

In [50]:
train_data.classes

['T-shirt/top',
 'Trouser',
 'Pullover',
 'Dress',
 'Coat',
 'Sandal',
 'Shirt',
 'Sneaker',
 'Bag',
 'Ankle boot']

In [ ]:
len(train_data.classes)

In [ ]:
train_data.class_to_idx# to see all the class and their indexes

In [ ]:
train_data.targets# gives the target tensor array

In [ ]:
# visualising the data
print(f"image-shape{image.shape}")

In [ ]:
plt.imshow(image.squeeze()) # if its b&w then why is there green channels here plt.imshow is for image nice.
plt.show()

In [ ]:
plt.imshow(image.squeeze(), cmap="gray")
plt.show()

In [ ]:
torch.manual_seed(42)
fig = plt.figure(figsize=(9,9))
rows, cols = 4, 4
for i in range(1, rows*cols+1):
    random_idx = torch.randint(0, len(train_data), size=[1]).item()
    img, label = train_data[random_idx]
    fig.add_subplot(rows, cols, i)
    plt.imshow(img.squeeze(), cmap="gray")
plt.show()

In [ ]:
from torch.utils.data import DataLoader
train_dataloader = DataLoader(
    dataset=train_data,
    batch_size=32,
    shuffle=True
)
test_dataloader = DataLoader(
    dataset=test_data,
    batch_size=32,
    shuffle=False
)

In [ ]:
print(f"Dataloader:{train_dataloader}\n")
print(f"Length of train dataloader:{len(train_dataloader)}\n")
print(f"Type of train dataloader:{type(train_dataloader)}\n")
print(f"Length of test dataloader:{len(test_dataloader)}\n")
print(f"Type of test dataloader:{type(test_dataloader)}\n")

In [ ]:
train_features_batch, train_labels_batch = next(iter(train_dataloader))
train_features_batch.shape, train_labels_batch.shape

In [ ]:
# sample
torch.manual_seed(42)# for reproducibility
random_idx = torch.randint(0,len(train_features_batch), size=[1]).item()# generating a random_idx of size 1 between 0 and total length of batch
img, label = train_features_batch[random_idx], train_labels_batch[random_idx]
plt.imshow(img.squeeze(), cmap="gray")
plt.show()
print(f"Label: {train_data.classes[label]}")

In [ ]:
from torch import nn
class FashionMNISTModelV0(nn.Module):
  def __init__(self, input_shape: int,hidden_units: int, output_shape: int):
     super().__init__()
     self.layer_stack = nn.Sequential(
         nn.Flatten(),
         nn.Linear(in_features=input_shape, out_features=hidden_units),
         nn.Linear(in_features=hidden_units, out_features=output_shape)
     )
  def forward(self, x):

    return self.layer_stack(x)



In [ ]:
torch.manual_seed(42)
model_0 = FashionMNISTModelV0(
    input_shape=784,
    hidden_units=10,
    output_shape=len(train_data.classes)
)
model_0

In [ ]:
import requests
from pathlib import Path

if Path('helper_functions.py').is_file():
    print('helper_functions.py already exists, skipping download')
else:
    print('Downloading helper_functions.py')
    request = requests.get('https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py')
    with open('helper_functions.py', 'wb') as f:
        f.write(request.content)

In [ ]:
from helper_functions import accuracy_fn



In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model_0.parameters(), lr=0.001)

In [ ]:
from timeit import default_timer as Timer# this is to keep track of how much time our neural network is taking
def print_train_time(start: float, end: float, device: torch.device = None):
    total_time = end - start
    print(f"Train time on {device}: {total_time:.3f} seconds")
    return total_time

In [ ]:
from functools import total_ordering
# now lets train it
from tqdm.auto import tqdm# for visualisation of speed of training just looks cool
torch.manual_seed(42)
train_time_start_on_cpu = Timer()# starting timer
for epoch in tqdm(range(10)):
    print(f"Epoch: {epoch}\n---------")
    train_loss = 0
    for batch, (X, y) in enumerate(train_dataloader):# batch is number of batches, then in one batch ew have X images with target y
        model_0.train()
        y_pred = model_0(X)
        loss = loss_fn(y_pred, y)
        if batch%400 ==0:
          print(loss)
        train_loss += loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if batch % 400 == 0:
            print(f"Looked at {batch * len(X)}/{len(train_dataloader.dataset)} samples")
    train_loss /= len(train_dataloader)
    test_loss, test_acc = 0, 0
    model_0.eval()
    with torch.inference_mode():
        for X, y in test_dataloader:
            test_pred = model_0(X)
            test_loss += loss_fn(test_pred, y)
            test_acc += accuracy_fn(y_true=y, y_pred=test_pred.argmax(dim=1))
        test_loss /= len(test_dataloader)
        test_acc /= len(test_dataloader)
    print(f"\nTrain loss: {train_loss} | Test loss: {test_loss}, Test acc: {test_acc}%\n")
train_time_end_on_cpu = Timer()
total_train_time_model_0 = print_train_time(start=train_time_start_on_cpu, end=train_time_end_on_cpu, device="cpu")

In [ ]:
print(y.shape, y.dtype)
print(y.min(), y.max())


In [ ]:
print(y_pred.shape)


In [ ]:
# making prediction
torch.manual_seed(42)
def eval_model(model: torch.nn.Module, data_loader: torch.utils.data.DataLoader, loss_fn: torch.nn.Module, accuracy_fn):
  loss,acc=0,0
  model.eval()
  with torch.inference_mode():
    for X,y in data_loader:
      y_pred = model(X)
      loss += loss_fn(y_pred, y)
      acc += accuracy_fn(y_true=y, y_pred=y_pred.argmax(dim=1))
    loss /= len(data_loader)
    acc /= len(data_loader)
  print(f"loss{loss}, acc{acc}")
# eval_model(model_0, test_dataloader, loss_fn, accuracy_fn)


In [ ]:
# now lets improve the accuracy and overall model performance
!nvidia-smi

In [ ]:
if torch.cuda.is_available():
  device = "cuda"
else:
  device = "cpu"
device

In [ ]:
from torch import nn
class FashionMNISTModelV0(nn.Module):
  def __init__(self, input_shape: int,hidden_units: int, output_shape: int):
     super().__init__()
     self.layer_stack = nn.Sequential(
         nn.Flatten(),
         nn.Linear(in_features=input_shape, out_features=hidden_units),
         nn.ReLU(),
         nn.Linear(in_features=hidden_units, out_features=hidden_units),
         nn.ReLU(),
         nn.Linear(in_features=hidden_units, out_features=output_shape)
     )
  def forward(self, x):

    return self.layer_stack(x)


In [ ]:
torch.manual_seed(42)
model_0 = FashionMNISTModelV0(
    input_shape=784,
    hidden_units=10,
    output_shape=len(train_data.classes)
).to(device)
model_0

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model_0.parameters(), lr=0.01)

In [ ]:
from functools import total_ordering
# now lets train it
from tqdm.auto import tqdm # for visualisation of speed of training just looks cool
torch.manual_seed(42)
train_time_start_on_cpu = Timer() # starting timer
for epoch in tqdm(range(10)):
    print(f"Epoch: {epoch}\n---------")
    train_loss = 0
    for batch, (X, y) in enumerate(train_dataloader):# batch is number of batches, then in one batch ew have X images with target y
        model_0.train()
        y_pred = model_0(X.to(device))
        loss = loss_fn(y_pred.to(device), y.to(device))

        train_loss += loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if batch % 400 == 0:
            print(f"Looked at {batch * len(X)}/{len(train_dataloader.dataset)} samples")
    train_loss /= len(train_dataloader)
    test_loss, test_acc = 0, 0
    model_0.eval()
    with torch.inference_mode():
        for X, y in test_dataloader:
            test_pred = model_0(X.to(device))
            test_loss += loss_fn(test_pred.to(device), y.to(device))
            test_acc += accuracy_fn(y_true=y.to(device), y_pred=test_pred.argmax(dim=1))
        test_loss /= len(test_dataloader)
        test_acc /= len(test_dataloader)
    print(f"\nTrain loss: {train_loss} | Test loss: {test_loss}, Test acc: {test_acc}%\n")
train_time_end_on_cpu = Timer()
total_train_time_model_0 = print_train_time(start=train_time_start_on_cpu, end=train_time_end_on_cpu, device="cpu")

In [ ]:
# making prediction
torch.manual_seed(42)
def eval_model(model: torch.nn.Module, data_loader: torch.utils.data.DataLoader, loss_fn: torch.nn.Module, accuracy_fn):
  loss,acc=0,0
  model.eval()
  with torch.inference_mode():
    for X,y in data_loader:
      y_pred = model(X.to(device))
      loss += loss_fn(y_pred, y.to(device))
      acc += accuracy_fn(y_true=y.to(device), y_pred=y_pred.argmax(dim=1))
    loss /= len(data_loader)
    acc /= len(data_loader)
  print(f"loss{loss}, acc{acc}")
eval_model(model_0, test_dataloader, loss_fn, accuracy_fn)

In [ ]:
# we got an better accuracy by introducing some non-linearity in model, we can functalise this also, but as we can see it not much different,
# so now we will do the same thing using convulational neural network, so we used linear, then non-linear, then also accuracy is not much , so conv is last hope🥹🥹


In [ ]:
import torch
from torch import nn
class FashionMNISTModelV2(nn.Module):
  def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
    super().__init__()
    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels=input_shape, out_channels=hidden_units, kernel_size=3, padding=1,stride=1),#2-d for 2-d layer
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3, padding=1,stride=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)

    )
    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3, padding=1,stride=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3, padding=1,stride=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)

    )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=hidden_units*7*7, out_features=output_shape)
    )
  def forward(self, x):
    x = self.conv_block_1(x)
    x = self.conv_block_2(x)
    x = self.classifier(x)
    return x

In [ ]:
image.shape

In [ ]:
torch.manual_seed(42)
model_2 = FashionMNISTModelV2(input_shape=1, hidden_units=10, output_shape=len(train_data.classes)).to(device)


In [ ]:
import torch
from torch import nn

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model_2.parameters(), lr=0.01)

In [ ]:
def train_step(model: torch.nn.Module, data_loader: torch.utils.data.DataLoader, loss_fn: torch.nn.Module, optimizer: torch.optim.Optimizer, device: torch.device):
  train_loss = 0
  train_acc=0
  for batch, (X, y) in enumerate(train_dataloader):# batch is number of batches, then in one batch ew have X images with target y
      model.train()
      X,y = X.to(device), y.to(device)
      y_pred = model(X.to(device))
      loss = loss_fn(y_pred.to(device), y.to(device))

      train_loss += loss
      train_acc += accuracy_fn(y_true=y.to(device), y_pred=y_pred.argmax(dim=1))
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()
      if batch % 400 == 0:
          print(f"Looked at {batch * len(X)}/{len(train_dataloader.dataset)} samples")
  train_loss /= len(train_dataloader)
  train_acc /= len(train_dataloader)
  print(f"Train loss: {train_loss} | Train acc: {train_acc}%\n")


In [ ]:
def test_step(model: torch.nn.Module, data_loader: torch.utils.data.DataLoader, loss_fn: torch.nn.Module, accuracy_fn, device: torch.device):
  test_loss, test_acc = 0, 0
  model.eval()
  with torch.inference_mode():

    for X, y in test_dataloader:
        X,y = X.to(device), y.to(device)
        test_pred = model(X.to(device))
        test_loss += loss_fn(test_pred.to(device), y.to(device))
        test_acc += accuracy_fn(y_true=y.to(device), y_pred=test_pred.argmax(dim=1))
    test_loss /= len(test_dataloader)
    test_acc /= len(test_dataloader)
    print(f" Test loss: {test_loss}, Test acc: {test_acc}%\n")


In [ ]:
# training our odel
from tqdm.auto import tqdm
torch.manual_seed(42)
torch.cuda.manual_seed(42)
from timeit import default_timer as Timer# this is to keep track of how much time our neural network is taking
train_time_start_model_2 = Timer()# starting timer
for epoch in tqdm(range(10)):
    print(f"Epoch: {epoch}\n---------")
    train_step(model=model_2, data_loader=train_dataloader, loss_fn=loss_fn, optimizer=optimizer, device=device)
    test_step(model=model_2, data_loader=test_dataloader, loss_fn=loss_fn, accuracy_fn=accuracy_fn, device=device)
train_time_end_model_2 = Timer()


In [ ]:
#best result so far
# using this model, and including some dataprocessing steps in starting we can make our model, useful for practical datasets
# here i am not doing that, and this is the end of the sos_25 for this side.